In [2]:
# Author: Arun Bamal
# Project: Data Engineering Assignmentimport duckdb

con = duckdb.connect("ecommerce.db")

print("Connected to DuckDB")

Connected to DuckDB


In [3]:
con.execute("""
SELECT 
    product_id,
    COUNT(*) AS views
FROM clean_events
WHERE event_type = 'view'
GROUP BY product_id
ORDER BY views DESC
LIMIT 10
""").fetchdf()

,product_id,views
0,1004856,419287
1,1004767,378777
2,1005115,327715
3,1004249,207422
4,1004833,203018
5,1005105,197930
6,1004870,190435
7,1002544,179249
8,4804056,179092
9,5100816,164608


In [4]:
con.execute("""
SELECT 
    product_id,
    SUM(price) AS revenue
FROM clean_events
WHERE event_type = 'purchase'
GROUP BY product_id
ORDER BY revenue DESC
LIMIT 10
""").fetchdf()

,product_id,revenue
0,1005115,12406807.35
1,1005105,10239248.68
2,1004249,6730112.92
3,1005135,5567806.64
4,1004767,5430723.43
5,1002544,4855702.39
6,1004856,3798956.42
7,1002524,3538299.12
8,1003317,3051294.26
9,1004870,3027098.05


In [5]:
con.execute("""
SELECT 
    category_code,
    SUM(price) AS total_revenue
FROM clean_events
WHERE event_type = 'purchase'
GROUP BY category_code
ORDER BY total_revenue DESC
LIMIT 10
""").fetchdf()

,category_code,total_revenue
0,electronics.smartphone,1.570496e+08
1,NaN,2.292494e+07
2,computers.notebook,8.979887e+06
3,electronics.video.tv,8.423408e+06
4,electronics.clocks,4.818305e+06
5,appliances.kitchen.washer,4.658647e+06
6,appliances.kitchen.refrigerators,3.830077e+06
7,electronics.audio.headphone,3.539127e+06
8,appliances.environment.vacuum,1.716425e+06
9,electronics.tablet,1.610974e+06


In [6]:
con.execute("""
SELECT 
    DATE(event_time) AS date,
    SUM(price) AS daily_revenue
FROM clean_events
WHERE event_type = 'purchase'
GROUP BY date
ORDER BY date
""").fetchdf()

,date,daily_revenue
0,2019-10-01,6275964.01
1,2019-10-02,6213628.53
2,2019-10-03,6233782.98
3,2019-10-04,8623684.47
4,2019-10-05,7341596.91
5,2019-10-06,6737660.78
6,2019-10-07,6348189.06
7,2019-10-08,6819832.28
8,2019-10-09,6855511.13
9,2019-10-10,6665600.86


In [7]:
con.execute("""
SELECT 
    brand,
    SUM(price) AS revenue
FROM clean_events
WHERE event_type = 'purchase'
    AND brand IS NOT NULL
GROUP BY brand
ORDER BY revenue DESC
LIMIT 10
""").fetchdf()

,brand,revenue
0,apple,1.112093e+08
1,samsung,4.640753e+07
2,xiaomi,9.194033e+06
3,huawei,4.883422e+06
4,acer,3.576720e+06
5,lg,3.387888e+06
6,lucente,3.124113e+06
7,sony,2.478197e+06
8,oppo,2.412960e+06
9,lenovo,1.752639e+06


In [8]:
con.execute("""
SELECT 
    user_id,
    COUNT(*) AS total_events
FROM clean_events
GROUP BY user_id
ORDER BY total_events DESC
LIMIT 10
""").fetchdf()

,user_id,total_events
0,512475445,7436
1,512365995,4013
2,526731152,2912
3,512505687,2894
4,513021392,2862
5,546159478,2433
6,546270188,2426
7,514649263,2390
8,516308435,2316
9,512401084,2232


In [9]:
con.execute("""
SELECT 
    event_type,
    COUNT(*) AS total
FROM clean_events
GROUP BY event_type
""").fetchdf()

,event_type,total
0,view,40779399
1,purchase,742849
2,cart,926516


In [10]:
con.execute("""
WITH funnel AS (
    SELECT 
        event_type,
        COUNT(*) AS total
    FROM clean_events
    GROUP BY event_type
)
SELECT 
    MAX(CASE WHEN event_type = 'view' THEN total END) AS views,
    MAX(CASE WHEN event_type = 'purchase' THEN total END) AS purchases,
    ROUND(
        100.0 * 
        MAX(CASE WHEN event_type = 'purchase' THEN total END) /
        MAX(CASE WHEN event_type = 'view' THEN total END),
        2
    ) AS conversion_rate_percent
FROM funnel
""").fetchdf()

,views,purchases,conversion_rate_percent
0,40779399,742849,1.82


In [11]:
con.close()